# PRODUCT PREDICTION

In [1]:
# Basic libraries
import pandas as pd
import numpy as np

# Date handling
from datetime import timedelta

In [2]:
# Load datasets

train = pd.read_excel(r"C:\Users\Administrator\Desktop\AUC\Customers_Transactions.xlsx")
test = pd.read_excel(r"C:\Users\Administrator\Desktop\AUC\Customers_Test_set.xlsx")

# Preview
print(train.head())
print(test.head())

  EventID  EventType ProductID                    ProductName  Quantity  \
0  612890  Purchased     97393  DIWALI LANTERN WITH CARDBOARD        32   
1  612890  Purchased    91668P        FLOWER POTS WITH FLOWER        32   
2  612890  Purchased    91668W             RED COLORED LIGHTS        32   
3  612890  Purchased     34386                    PHOTO FRAME        68   
4  612890  Purchased     33577                      FUNNY HAT        44   

        EventDateTime  UnitPrice  UserID  
0 2019-12-01 07:45:00     624.80   25430  
1 2019-12-01 07:45:00     607.40   25430  
2 2019-12-01 07:45:00     607.40   25430  
3 2019-12-01 07:45:00     197.85   25430  
4 2019-12-01 07:45:00     123.90   25430  
  EventID  EventType ProductID                      ProductName  Quantity  \
0  659825  Purchased     34101                      CAR CLEANER        13   
1  659846  Purchased     35286     CHRISTMAS LIGHTS 10 REINDEER        12   
2  659846  Purchased     35305  BAMBOO CUTTING BOARD COLLECT

In [3]:
# Convert datetime column
train['EventDateTime'] = pd.to_datetime(train['EventDateTime'])
test['EventDateTime'] = pd.to_datetime(test['EventDateTime'])

# Check event types
print(train['EventType'].value_counts())

EventType
Purchased    407695
Returned       9839
Name: count, dtype: int64


In [4]:
# Assign weights to actions
train['Score'] = train['EventType'].map({
    'Viewed': 1,
    'AddToCart': 3,
    'Purchased': 5
})

In [5]:
# Create preference matrix
user_product = train.pivot_table(
    index='UserID',
    columns='ProductName',
    values='Score',
    aggfunc='sum',
    fill_value=0
)

print(user_product.head())

ProductName    DOORMAT UNION JACK GUNS AND ROSES   3 STRIPEY MICE FELTCRAFT  \
UserID                                                                        
24691                                        0.0                        0.0   
24692                                        0.0                        0.0   
24693                                        0.0                        0.0   
24694                                        0.0                        0.0   
24696                                        0.0                        0.0   

ProductName   4 PURPLE FLOCK DINNER CANDLES   ANIMAL STICKERS  \
UserID                                                          
24691                                   0.0               0.0   
24692                                   0.0               0.0   
24693                                   0.0               0.0   
24694                                   0.0               0.0   
24696                                   0.0             

In [7]:
train['Score'] = train['EventType'].map({
    'Viewed': 1,
    'AddToCart': 3,
    'Purchased': 5
})

In [8]:
user_product_score = train.groupby(['UserID','ProductName'])['Score'].sum().reset_index()

In [9]:
top_products = user_product_score.loc[
    user_product_score.groupby('UserID')['Score'].idxmax()
]

In [10]:
top_products = top_products[['UserID','ProductName']]
top_products.columns = ['UserID','Recommended_Product']

In [11]:
test = test.merge(top_products, on='UserID', how='left')

In [12]:
popular_product = train['ProductName'].value_counts().idxmax()

test['Recommended_Product'] = test['Recommended_Product'].fillna(popular_product)

In [13]:
print(test[['UserID','Recommended_Product']].head())

   UserID      Recommended_Product
0   25392             WATER BOTTLE
1   29856  ROSE GOLD VANITY MIRROR
2   29856  ROSE GOLD VANITY MIRROR
3   29856  ROSE GOLD VANITY MIRROR
4   29856  ROSE GOLD VANITY MIRROR


# TIME PREDICTION

In [21]:
# Find last interaction per user
last_activity = train.groupby('UserID')['EventDateTime'].max().reset_index()

# Rename column
last_activity.columns = ['UserID','Last_Activity']

In [22]:
print(last_activity.head())

   UserID       Last_Activity
0   24691 2020-10-04 16:33:00
1   24692 2020-12-07 14:57:00
2   24693 2020-09-27 14:59:00
3   24694 2020-10-28 08:23:00
4   24696 2020-11-29 15:23:00


In [23]:
test = test.merge(last_activity, on='UserID', how='left')

In [34]:
test['Last_Activity'] = test['Last_Activity'].fillna(test['EventDateTime'])

In [45]:
popular_product = train['ProductName'].value_counts().idxmax()
test['Recommended_Product'] = test['Recommended_Product'].fillna(popular_product)

In [46]:
print(test.columns)

Index(['EventID', 'EventType', 'ProductID', 'ProductName', 'Quantity',
       'EventDateTime', 'UnitPrice', 'UserID', 'Recommended_Product',
       'Last_Activity_x', 'Predicted_Purchase_Date', 'Last_Activity_y',
       'Last_Activity'],
      dtype='object')


In [47]:
# Filter only purchase events
purchase = train[train['EventType'] == 'Purchased']

In [48]:
first_purchase = purchase.groupby('UserID')['EventDateTime'].min().reset_index()
first_purchase.columns = ['UserID','First_Purchase']

In [49]:
# Merge activity + purchase
time_data = last_activity.merge(first_purchase, on='UserID')

# Calculate days
time_data['Days_to_Purchase'] = (
    time_data['First_Purchase'] - time_data['Last_Activity']
).dt.days

In [50]:
# Remove negative values
time_data = time_data[time_data['Days_to_Purchase'] >= 0]

In [51]:
avg_days = int(time_data['Days_to_Purchase'].mean())

print("Average Days:", avg_days)

Average Days: 0


In [52]:
test['Predicted_Purchase_Date'] = test['Last_Activity'] + pd.to_timedelta(avg_days, unit='D')

# TARGETING SYSTEM

In [53]:
product_user_score = train.groupby(['ProductName','UserID'])['Score'].sum().reset_index()

In [56]:
def get_target_users(product_name, top_n=5):

    data = product_user_score[product_user_score['ProductName'] == product_name]

    if data.empty:
        return "No data available"

    top_users = data.sort_values(by='Score', ascending=False).head(top_n)

    result = top_users.merge(
        test[['UserID','Predicted_Purchase_Date']],
        on='UserID',
        how='left'
    )

    result['Predicted_Purchase_Date'] = result['Predicted_Purchase_Date'].fillna("Unknown")

    return result[['UserID','Score','Predicted_Purchase_Date']]

In [55]:
get_target_users("ROSE GOLD VANITY MIRROR")

,UserID,Score,Predicted_Purchase_Date
0,30195,360.0,2020-12-02 15:27:00
1,30195,360.0,2020-12-02 15:27:00
2,30195,360.0,2020-12-02 15:27:00
3,30195,360.0,2020-12-02 15:27:00
4,30195,360.0,2020-12-02 15:27:00
...,...,...,...
532,28113,115.0,2020-08-20 15:58:00
533,28113,115.0,2020-08-20 15:58:00
534,28113,115.0,2020-08-20 15:58:00
535,28113,115.0,2020-08-20 15:58:00


In [58]:
final_output = test[['UserID','Recommended_Product','Predicted_Purchase_Date']]

final_output.to_excel(r"C:\Users\Administrator\Desktop\AUC\Final_Predictions.xlsx", index=False)